[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [asyncpg and psycopg3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)

# AsyncConnection


## What you will be able to do

Use psycopg with `await`, and say what a coroutine is before it is awaited and what happens to one
that never is. Run queries at the same time with `asyncio.gather`, and say why two tasks sharing one
connection do not go faster. Cancel a query from Python and have the server stop working on it. Put
a time limit on a statement. And recognize the three mistakes that come from mixing the synchronous
and asynchronous halves of the same library.


## The idea

### The problem

`await` is not a speed feature. It is a way of saying "this will wait, let something else run", and
it buys you nothing unless there is something else that could run and something real to wait for.

The mistake this notebook exists for is subtle: a program moves to `AsyncConnection`, wraps its
queries in `asyncio.gather`, measures nothing, and runs at exactly the speed it did before, because
every task is queued behind the same connection. One connection is one session and holds one
statement at a time, and `await` cannot change that.

### What a coroutine is

Calling an `async def` function does not run it. It gives you a coroutine, which is a description of
work that has not started. `await` is what starts it and waits for it. `asyncio.gather` is what
starts several and waits for all of them.

A notebook already has an event loop running, which is why every cell below can `await` at the top
level with no `asyncio.run` anywhere. A script does not, which is why the first look has one.

### Why it works that way

A PostgreSQL connection is one session speaking one protocol conversation. Two statements cannot be
in flight on it at once, in any driver, in any language. So concurrency in a database program is
concurrency across connections, and everything in **Connection Pools** follows from that.

What `await` buys is that the waiting is yielded rather than blocked: while one connection waits for
the server, the event loop can run something else, including another connection's query.

### Where this shows up

Any web application that talks to a database. The request handler that takes fifty milliseconds of
database time can serve other requests during it, which is the whole argument for asynchronous code
here, and it needs a connection per concurrent query to be true.

### What this notebook covers

The coroutine and the `await`. `gather`, and the measurement that shows sharing a connection does
not work. Async cursors, server-side cursors and `COPY`, which are the same as their synchronous
versions with `async` in front. Cancellation and `statement_timeout`. Then the four failures, which
are all about the seam between the two halves of psycopg.

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import asyncio
import time

import psycopg


async def ask(conn, label):
    cur = await conn.execute("SELECT pg_sleep(0.3), %s", (label,))
    return (await cur.fetchone())[1]


async def main():
    one = await psycopg.AsyncConnection.connect("dbname=guide")
    two = await psycopg.AsyncConnection.connect("dbname=guide")

    start = time.perf_counter()
    await asyncio.gather(ask(one, "a"), ask(one, "b"))      # both tasks on one connection
    shared = time.perf_counter() - start

    start = time.perf_counter()
    await asyncio.gather(ask(one, "a"), ask(two, "b"))      # one connection each
    separate = time.perf_counter() - start

    print(f"two tasks, one connection:  {shared:.1f}s  <- they took turns")
    print(f"two tasks, two connections: {separate:.1f}s  <- they overlapped")
    await one.close()
    await two.close()


asyncio.run(main())                             # a script needs this, a notebook does not
```

```
two tasks, one connection:  0.6s  <- they took turns
two tasks, two connections: 0.3s  <- they overlapped
```

Two queries of three hundred milliseconds each. On one connection they take six hundred, because the
second one could not start until the first had finished. On two they take three hundred, because
that is what waiting at the same time means. Nothing in the `gather` changed.


## Setup

Eleven imports, psycopg, the server, and three helpers.

- `psycopg` is the driver, and `errors` is the exception classes
- `asyncio` is the standard library's event loop, `time` measures, and `warnings` and `gc` are for
  catching one warning without the file path it carries
- `subprocess`, `sys`, `os`, `getpass` stand the server up with `version` and `PackageNotFoundError`

Three helpers. `ask` runs one slow query, so that waiting is long enough to see. `rounded` prints a
tenth of a second, which is as precise as a timing on a shared machine can honestly be, and the
sleeps here are long enough that the tenths are the same on every run. `warned_about` catches a
warning rather than letting it print, because Python attaches a file path to it.


In [1]:
import asyncio
import gc
import getpass
import os
import subprocess
import sys
import time
import warnings
from importlib.metadata import PackageNotFoundError, version

try:
    if version("psycopg") < "3.3" or version("asyncpg") < "0.31":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "psycopg[binary,pool]==3.3.6", "psycopg-pool==3.3.2", "asyncpg==0.31.0"],
                   check=True)

import psycopg
from psycopg import errors

def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(database="postgres"):
    """Whether a server is there, asked the only way that needs no client binaries."""
    try:
        with psycopg.connect(f"dbname={database}", connect_timeout=2):
            return True
    except psycopg.OperationalError:
        return False


def start_server(wait=60):
    """Install and start PostgreSQL if nothing is answering. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No PostgreSQL is answering. Start your own server and run this again: "
                           "this cell only installs one on Linux, which is what Colab runs.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"{sudo}apt-get -qq update")
    shell(f"{sudo}apt-get -qq -y install postgresql postgresql-contrib")
    shell(f"{sudo}service postgresql start")                        # Colab has no systemd

    for attempt in range(1, wait + 1):                              # start returns before it listens
        if shell("pg_isready -q")[0] == 0:
            break
        print(f"  waiting for the cluster ({attempt})")              # a silent minute looks hung
        time.sleep(1)
    else:
        raise RuntimeError(f"PostgreSQL did not accept connections within {wait} seconds.")

    me = getpass.getuser()                                          # peer authentication wants a role
    asking = f"""sudo -u postgres psql -tAc "SELECT 1 FROM pg_roles WHERE rolname='{me}'" """
    if shell(asking)[1] != "1":                                     # named for the operating system user
        shell(f"sudo -u postgres createuser -s {me}")
    return "installed and started"

def build(rows=5000):
    """Make the guide database and its events table, and fill it once."""
    with psycopg.connect("dbname=postgres", autocommit=True) as conn:
        if not conn.execute("SELECT 1 FROM pg_database WHERE datname = 'guide'").fetchone():
            conn.execute("CREATE DATABASE guide")                   # cannot run in a transaction

    with psycopg.connect("dbname=guide", autocommit=True) as conn:
        for (leftover,) in conn.execute(                            # whatever an earlier run made
                "SELECT tablename FROM pg_tables "
                "WHERE schemaname = 'public' AND tablename <> 'events'").fetchall():
            conn.execute(f'DROP TABLE IF EXISTS "{leftover}" CASCADE')

        conn.execute("""CREATE TABLE IF NOT EXISTS events (
                            id bigserial PRIMARY KEY,
                            ts timestamptz NOT NULL DEFAULT now(),
                            kind text NOT NULL,
                            payload jsonb NOT NULL)""")
        if conn.execute("SELECT count(*) FROM events").fetchone()[0] == 0:
            conn.execute("""INSERT INTO events (kind, payload)
                            SELECT (ARRAY['click', 'view', 'purchase'])[1 + n %% 3],
                                   jsonb_build_object('n', n, 'size', 1 + n %% 7)
                            FROM generate_series(1, %s) AS n""", (rows,))
        return conn.execute("SELECT count(*) FROM events").fetchone()[0]

def report():
    """One line naming what this notebook is running against."""
    rows = build()                                                  # makes the database if it is new
    with psycopg.connect("dbname=guide") as conn:
        major = int(conn.execute("SHOW server_version_num").fetchone()[0]) // 10000
    return (f"PostgreSQL {major} | psycopg {version('psycopg')} | asyncpg {version('asyncpg')} "
            f"| events: {rows} rows")

async def ask(conn, label, seconds=0.3):
    """One slow query, so that waiting is visible."""
    cur = await conn.execute("SELECT pg_sleep(%s), %s", (seconds, label))
    return (await cur.fetchone())[1]


def rounded(seconds):
    """A tenth of a second, which is as precise as a timing here can honestly be."""
    return f"{seconds:.1f}s"


async def warned_about(work):
    """Run something that warns, and give back the warnings without the file paths they carry."""
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        await work()
        gc.collect()                                          # a lost coroutine warns when collected
    return [f"{w.category.__name__}: {w.message}" for w in caught]


print("server:", start_server())
print(report())
print("an event loop is running already:", asyncio.get_running_loop().is_running())


server: already running
PostgreSQL 16 | psycopg 3.3.6 | asyncpg 0.31.0 | events: 5000 rows
an event loop is running already: True


## Worked examples

### A coroutine, and awaiting it

Calling the function is not running it:


In [2]:
conn = await psycopg.AsyncConnection.connect("dbname=guide")

pending = conn.execute("SELECT 1")                                  # no await
print("what execute gave back:", type(pending).__name__)

cur = await pending
print("after awaiting it:      ", await cur.fetchone())


what execute gave back: coroutine
after awaiting it:       (1,)


`await psycopg.AsyncConnection.connect(...)` is worth reading closely: the connecting is itself
something to wait for, so it is awaited, and the result is the connection.

A coroutine that is never awaited is a bug that Python reports at collection time rather than where
it happened:


In [3]:
async def forgets():
    conn.execute("SELECT 1")                                        # no await, so it never runs


for line in await warned_about(forgets):
    print(line)


That warning is caught here rather than printed loose, because Python attaches the file and line to
it. In your own code it appears on the console, usually long after the line that caused it, and it
means a query you wrote never ran.

### Two tasks, one connection

The first look measured it. Here is why, asked of the connection:


In [4]:
async def timed(*work):
    start = time.perf_counter()
    await asyncio.gather(*work)
    return time.perf_counter() - start


other = await psycopg.AsyncConnection.connect("dbname=guide")

print("one connection, two tasks: ", rounded(await timed(ask(conn, "a"), ask(conn, "b"))))
print("two connections, two tasks:", rounded(await timed(ask(conn, "a"), ask(other, "b"))))
third = await psycopg.AsyncConnection.connect("dbname=guide")
print("three connections, three:  ",
      rounded(await timed(ask(conn, "a"), ask(other, "b"), ask(third, "c"))))
await third.close()


one connection, two tasks:  0.6s
two connections, two tasks: 0.3s
three connections, three:   0.3s


The third line is the point of the whole exercise: three queries of three hundred milliseconds each
finish in three hundred milliseconds when each has a connection of its own. That is the thing
asynchronous code buys, and it scales with connections rather than with tasks.

This is also the first hint of why **Connection Pools** exists: if concurrency needs a connection
each, something has to manage how many there are.

### Everything else, with async in front

The cursor, the server-side cursor and `COPY` are the same objects with `async` added:


In [5]:
async with conn.cursor() as cur:
    await cur.execute("SELECT count(*) FROM events")
    print("an async cursor:", await cur.fetchone())

async with conn.cursor(name="big") as cur:                          # server-side, as before
    await cur.execute("SELECT n FROM generate_series(1, 10) AS n")
    print("an async server cursor:", [row[0] async for row in cur])


an async cursor: (5000,)
an async server cursor: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


`async for` is the loop over an asynchronous iterator, and it is what a server-side cursor needs
because each batch is a round trip.

`COPY` is the same shape too, with `await` on the write:


In [6]:
await conn.execute("DROP TABLE IF EXISTS async_loaded")
await conn.execute("CREATE TABLE async_loaded (id int, note text)")

async with conn.cursor() as cur:
    async with cur.copy("COPY async_loaded (id, note) FROM STDIN") as copy:
        for number in range(5):
            await copy.write_row((number, "written asynchronously"))

await conn.commit()                                      # or the rollback further down undoes it all
cur = await conn.execute("SELECT count(*) FROM async_loaded")
print("rows:", await cur.fetchone())


rows: (5,)


### Stopping a query

Two ways, and they are different. `statement_timeout` is the server's: it gives up on its own after
the time you set:


In [7]:
await conn.execute("SET statement_timeout = '300ms'")

try:
    await conn.execute("SELECT pg_sleep(5)")
except errors.QueryCanceled as error:
    print(type(error).__module__ + "." + type(error).__name__ + ":", error)

await conn.rollback()
print("the connection after a rollback:", conn.info.transaction_status.name)
await conn.execute("SET statement_timeout = 0")                     # back to no limit
await conn.commit()


psycopg.errors.QueryCanceled: canceling statement due to statement timeout
the connection after a rollback: IDLE


Canceling the task is Python's, and it reaches the server too: psycopg sends a cancellation, so the
query stops rather than running on unwatched:


In [8]:
start = time.perf_counter()
task = asyncio.create_task(conn.execute("SELECT pg_sleep(5)"))
await asyncio.sleep(0.3)
task.cancel()

try:
    await task
except asyncio.CancelledError:
    print("the query ran for", rounded(time.perf_counter() - start), "rather than 5 seconds")

print("the connection afterwards:", conn.info.transaction_status.name)
await conn.rollback()
print("after a rollback:        ", conn.info.transaction_status.name)


the query ran for 0.3s rather than 5 seconds
the connection afterwards: INERROR
after a rollback:         IDLE


The connection is left in an error state, which is the **Transactions and Errors** rule arriving
here: a statement that did not finish leaves a transaction that has to be rolled back. A cancellation
is a failure as far as the transaction is concerned.

### When to reach for which

| What you want | How to write it |
|---|---|
| a connection | `await psycopg.AsyncConnection.connect(...)` |
| a query | `cur = await conn.execute(...)`, then `await cur.fetchone()` |
| several queries at once | `asyncio.gather`, with a connection each |
| a loop over a large result | `async for row in cur` on a named cursor |
| a bulk load | `async with cur.copy(...)` and `await copy.write_row(...)` |
| a time limit the server enforces | `SET statement_timeout` |
| to stop waiting, and stop the query | cancel the task |
| to run this in a script | `asyncio.run(main())` around it |
| to run it in a notebook | nothing, `await` works at the top level |

The default is one connection per concurrent piece of work. Sharing a connection between tasks is
not faster and is not safer, and the notebook after the next one is about the object that hands them
out.

### A page that asks three things at once, finished

Everything above, as the shape a request handler has: several queries that do not depend on each
other, each on its own connection, awaited together.


In [9]:
async def page():
    """Three independent questions, asked at the same time."""
    connections = [await psycopg.AsyncConnection.connect("dbname=guide") for _ in range(3)]

    async def counted(conn, sql):
        cur = await conn.execute(sql)
        return (await cur.fetchone())[0]

    try:
        events, kinds, biggest = await asyncio.gather(
            counted(connections[0], "SELECT count(*) FROM events"),
            counted(connections[1], "SELECT count(DISTINCT kind) FROM events"),
            counted(connections[2], "SELECT max((payload ->> 'size')::int) FROM events"))
        return {"events": events, "kinds": kinds, "biggest": biggest}
    finally:
        for connection in connections:
            await connection.close()


print(await page())


{'events': 5000, 'kinds': 3, 'biggest': 7}


Three queries, three connections, one wait. On a page where each query took fifty milliseconds this
is the difference between a hundred and fifty milliseconds and fifty, and it is the reason to write
a handler this way rather than the reason to write the whole program asynchronously.

### Where each part came from

| In the page | What it relies on | The section that showed it |
|---|---|---|
| a connection per query | one statement at a time per session | Two tasks, one connection |
| `asyncio.gather` | several coroutines started together | A coroutine, and awaiting it |
| `await conn.execute(...)` | the query, yielded rather than blocked | A coroutine, and awaiting it |
| the `finally` closing each one | connections not left open on a failure | **Connecting and Executing** |
| `(payload ->> 'size')::int` | a JSON value as a number | **JSONB** |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/10-async-connection-solutions.ipynb).

**1.** Open an asynchronous connection, run one query, and print what `execute` gives back before
and after it is awaited.


In [10]:
# your code here


**2.** Time two slow queries on one connection and the same two on two connections.


In [11]:
# your code here


**3.** Read ten rows through an asynchronous server-side cursor with `async for`.


In [12]:
# your code here


**4.** Set a statement timeout and run a query that exceeds it, then put the connection back in
order.


In [13]:
# your code here


**5.** Start a slow query as a task, cancel it, and show it stopped sooner than it would have
finished.


In [14]:
# your code here


**6.** Ask three different counts at the same time, each on its own connection.


In [15]:
# your code here


## Common errors

### RuntimeError: asyncio.run() cannot be called from a running event loop


In [16]:
async def counted():
    conn = await psycopg.AsyncConnection.connect("dbname=guide")
    cur = await conn.execute("SELECT count(*) FROM events")
    total = await cur.fetchone()
    await conn.close()
    return total


asyncio.run(counted())


RuntimeError: asyncio.run() cannot be called from a running event loop

A notebook cell is already running inside an event loop, and a loop cannot be started inside a loop.
This is the first thing that happens to anybody who copies asynchronous code out of a script and into
a notebook.

The fix is to drop the wrapper, because the thing it provides is already there:


In [17]:
print("just await it:", await counted())


just await it: (5000,)


There is a library that patches the loop to allow the nesting, and it is worth saying plainly that
you do not need it here and it is not recommended: in a notebook, `await` at the top level of a cell
is the supported way, and in a script `asyncio.run` is.

### TypeError: 'Connection' object can't be awaited


In [18]:
await psycopg.connect("dbname=guide")


TypeError: 'Connection' object can't be awaited

`psycopg.connect` is the synchronous one. It returns a connection rather than something to wait for,
and awaiting an ordinary object is a Python error rather than a psycopg one.

The two live side by side in the same package, which is convenient and is exactly why this happens:


In [19]:
synchronous = psycopg.connect("dbname=guide")
print("psycopg.connect                ->", type(synchronous).__name__)
synchronous.close()

asynchronous = await psycopg.AsyncConnection.connect("dbname=guide")
print("await AsyncConnection.connect  ->", type(asynchronous).__name__)
await asynchronous.close()


psycopg.connect                -> Connection
await AsyncConnection.connect  -> AsyncConnection


### RuntimeWarning: coroutine 'AsyncConnection.execute' was never awaited


In [20]:
async def forgot():
    conn = await psycopg.AsyncConnection.connect("dbname=guide")
    conn.execute("INSERT INTO async_loaded VALUES (99, 'this never ran')")   # no await
    await conn.commit()
    await conn.close()


for line in await warned_about(forgot):
    print(line)

cur = await conn.execute("SELECT count(*) FROM async_loaded WHERE id = 99")
print("rows with id 99:", await cur.fetchone())


rows with id 99: (0,)


Two warnings, and only one of them came from this cell. The second is the coroutine that
`asyncio.run` was given several cells ago and never got to, surfacing here because here is where
Python happened to collect it. That is the whole difficulty in one line: a missing `await` is the
quietest bug in asynchronous code, the statement never ran, nothing raised where it was written, and
the warning shows up somewhere else entirely.

It is worth knowing that this is what a benchmark showing asynchronous code as impossibly fast
usually is, which is the subject of **Which Driver**. The habit that prevents it is to await at the
point of the call, every time, and to treat the warning as an error in development:


In [21]:
async def remembered():
    conn = await psycopg.AsyncConnection.connect("dbname=guide")
    await conn.execute("INSERT INTO async_loaded VALUES (99, 'this one ran')")
    await conn.commit()
    await conn.close()


await remembered()
checker = await psycopg.AsyncConnection.connect("dbname=guide")
cur = await checker.execute("SELECT count(*) FROM async_loaded WHERE id = 99")
print("now it is there:", await cur.fetchone())
await checker.close()


now it is there: (1,)


### No error, and no speedup: two tasks sharing one connection


In [22]:
shared = await psycopg.AsyncConnection.connect("dbname=guide")
separate = await psycopg.AsyncConnection.connect("dbname=guide")

print("four tasks, one connection: ",
      rounded(await timed(*(ask(shared, str(n)) for n in range(4)))))
print("four tasks, two connections:",
      rounded(await timed(ask(shared, "a"), ask(separate, "b"),
                          ask(shared, "c"), ask(separate, "d"))))
await shared.close()
await separate.close()


four tasks, one connection:  1.2s
four tasks, two connections: 0.6s


Nothing failed, every query returned, and the first line took four times as long as one query. The
tasks were concurrent in Python and serialized at the connection, which is the one thing `gather`
cannot fix.

Halving the connections halves the time, and the pattern generalizes: the number that matters is how
many connections there are, not how many tasks. A pool is the object that owns that number, and it
is what **Connection Pools** is about.


In [23]:
for connection in (conn, other):
    await connection.close()
print("connections closed")


connections closed


## Recap

- Calling an `async def` gives a coroutine, which has not run. `await` runs it, and a coroutine that
  is never awaited warns at collection time and does nothing.
- A notebook has an event loop already, so `await` works at the top level of a cell and
  `asyncio.run` raises. A script is the other way round.
- One connection holds one statement at a time. Tasks sharing a connection take turns, and
  `asyncio.gather` cannot change that. Concurrency needs a connection each.
- Cursors, server-side cursors and `COPY` are the same objects with `async` in front, and `async for`
  is the loop over a server-side cursor.
- `SET statement_timeout` makes the server give up. Canceling the task makes psycopg tell the server
  to stop, and leaves the transaction needing a rollback.
- `psycopg.connect` is synchronous and `psycopg.AsyncConnection.connect` is not, and awaiting the
  first is a `TypeError` rather than anything to do with the database.


## What is next

The **asyncpg** notebook is the other driver, on its own terms: `$1` instead of `%s`, `Record`
instead of a row factory, and the transaction model that catches everybody, where a statement run
outside a block is committed the moment it returns.


---

&#8592; **Previous:** [Pipeline Mode](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/09-pipeline-mode.ipynb)  &nbsp;·&nbsp;  [asyncpg and psycopg3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/asyncpg-and-psycopg3-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [asyncpg](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/asyncpg-and-psycopg3-deep-dive/11-asyncpg.ipynb) &#8594;
